In this code we will look at the gate we obtain from 'nn_tutorial.ipynb' and compute its fidelity using qutip. This is a nice sanity check, and we can also use qutip later to e.g. analyse the performance of the gates in presence of fluctuations on the controls as well as decay. To start off we can compute the Bell state fidelity with the $CZ$ gate. (Note to self: look for a nice test for the CPHASE gates)

In [5]:
# can start off importing 
import os 

os.chdir("/Users/mmohan/Library/CloudStorage/OneDrive-TUEindhoven/Pasqal_/parametrized_gates/time_optimal_phase_og/")

import qutip as qt 
import numpy as np 
import torch 
import cst_n_fn as cfn 
from schsolve import schsolver, neural_trainer 
import torchdiffeq as tdf 
import const_czphi as czphi 

device = 'cpu'

# file handling section start 
# we should also temporarily set the variable angle_batch in const_czphi.py to 1
# indeed test using cz_762_optimal to see if F = 1 for my code == F = 1 for qutip 

#dict_ = torch.load('data/rand_models/net_adam_10_4')

dict_ = torch.load('data/final_models/pt5pi_to_pi')
network = dict_['network']
angle = torch.pi 

#input_tensor = torch.cat([torch.ones(czphi.time_steps,1)*angle, time_arr], dim = -1)
input_tensor = torch.tensor([[torch.pi]]).reshape(1,1)
#pred_outputs_rabi = ((network(input_tensor).select(1,0))*\
#        (czphi.range_rabi[1] - czphi.range_rabi[0]) + czphi.range_rabi[0])

pred_outputs_det = ((network(input_tensor))*\
        (czphi.range_detuning[1] - czphi.range_detuning[0]) + czphi.range_detuning[0])



In [1]:
#rabi_arr = pred_outputs_rabi # so for pi 
#det_arr = pred_outputs_det

rabi_arr = cfn.list_to_fn_tensor(pred_outputs_rabi, czphi.gatetime, czphi.time_steps) 
det_arr = cfn.list_to_fn_tensor(pred_outputs_det, czphi.gatetime, czphi.time_steps)
    
czphi.instance.hamiltonian.rabi_tensored["pulse 0"] = \
        czphi.instance.hamiltonian.rabi_tensored["pulse 1"] = rabi_arr
czphi.instance.hamiltonian.det_tensored["pulse 0"] = \
        czphi.instance.hamiltonian.det_tensored["pulse 1"] = det_arr  
    
    # ! tasks: still need to solve multiple ODEs instead of one, natively     
sol_intrm = czphi.reduce_r_dim_2q_vector(tdf.odeint(czphi.instance,\
                czphi.init_matrix[0].reshape(1, 9, 9), torch.linspace(0, czphi.gatetime, czphi.time_steps).to(device), 
                method = 'dopri5')[-1])

phi_01 = torch.angle(sol_intrm[:, 1, 1]).item()

#phi_01 = 2.1788077354431152

one_r = qt.basis(3, 1)*qt.basis(3, 2).dag()

h_1q_rabi = 0.5*(one_r + one_r.dag()) # * factor of 0.5 for the rabi frequency 

h_2q_rabi = qt.tensor(h_1q_rabi, qt.qeye(3)) + qt.tensor(qt.qeye(3), h_1q_rabi)

h_1q_det = qt.basis(3, 2)*qt.basis(3, 2).dag() # * detuning on the 1 - r transition 

h_2q_det = qt.tensor(h_1q_det, qt.qeye(3)) + qt.tensor(qt.qeye(3), h_1q_det)

# * interaction hamiltonian below

v_int = 21.1*cfn.rabi
rr = qt.tensor(qt.basis(3, 2), qt.basis(3, 2))
h_2q_int = rr*rr.dag()
# ? is there also a minus term for the detuning? 
# first we prepare a double Hadamard gate 
init_ = (1/np.sqrt(2))*(qt.basis(3, 0) + qt.basis(3, 1))
init_ = qt.tensor(init_, init_)
# * 
hamiltonian = [[h_2q_rabi, pred_outputs_rabi.detach().numpy()], [h_2q_det, pred_outputs_det.detach().numpy()], v_int*h_2q_int]
soln = qt.mesolve(hamiltonian, init_, np.linspace(0, czphi.gatetime, czphi.time_steps))
soln = soln.states[-1].eliminate_states([2,5,6,7,8]) # * remove the rydberg states 
soln.dims = [[2,2], [1]]

# basically using torch.angle(soln[1,1] or soln[2,2]) -- as pulse is symmetric, phi_01 = phi_10 

oneq_rot = qt.Qobj(cfn.corr_1q_rotation(torch.tensor(phi_01)).detach().numpy())
oneq_rot.dims = [[2,2], [2,2]]
soln = oneq_rot*soln 
hadamard = qt.Qobj(((1/np.sqrt(2))*np.array([[1, 1],[1, -1]])))
had_op = qt.tensor(qt.qeye(2), hadamard)
soln = had_op*soln 
print("Fidelity of bell state preparation: ")
zer = qt.basis(2, 0)
one = qt.basis(2, 1)
bell = (1/np.sqrt(2))*(qt.tensor(zer, zer) + qt.tensor(one, one))
print(qt.fidelity(bell, soln)**2)



NameError: name 'cfn' is not defined

In [ ]:
time_arr = torch.linspace(0, czphi.gatetime, czphi.time_steps).reshape((czphi.time_steps, 1))

desired_angle = torch.FloatTensor(czphi.angle_batch, 1).uniform_(czphi.angle_domain[0], czphi.angle_domain[1]).to(device)
ideal_stack_epoch = cfn.czp_gate_stack(desired_angle)
desired_angle_expanded = desired_angle.repeat(1, czphi.time_steps).view(-1, 1)
input_tensor = torch.cat([desired_angle_expanded, time_arr], dim=-1)
